# LLM Evaluation Concepts

This is Day 2 of Week 1. Prompt changes without measurement are guesses. LLM evaluation is the discipline of turning subjective output quality into trackable metrics — the same regression discipline applied to code, but adapted for stochastic, natural-language outputs. This notebook covers the three-layer evaluation stack: (1) **unit evals** — deterministic checks on output structure and format; (2) **metric-based evals** — quantitative scores for faithfulness, answer relevance, and context recall using RAGAS; (3) **LLM-as-judge** — using a strong model to score outputs where reference answers are unavailable. We build a lightweight eval harness that can run these checks automatically on any prompt change.

The setup mirrors [notebook 03](/courses/llm-eng/03-rag-concepts.html): we reuse the same 10-sentence financial corpus and the same `LLMClient` class, so the eval machinery we build here slots directly into the RAG pipeline we already have.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Why Evaluation Matters

**The regression risk.** Changing "You are a financial analyst" to "You are a helpful assistant" in a system prompt can silently drop accuracy by 30% on domain-specific tasks. Without a test suite you will not know until a compliance officer flags a hallucinated regulatory figure in a client report — at which point the cost is not a failed unit test but a potential regulatory action.

<br>

**The eval set is your test suite.** In traditional software, a change that breaks a function fails immediately and loudly. In an LLM system, a broken prompt degrades outputs gradually and silently — the model still returns text, just worse text. An eval set that runs on every prompt commit is the equivalent of `pytest`: it makes regressions visible before they reach production.

<br>

**The cost of missing an eval.** Consider a RAG pipeline that answers questions about SEC filings. A hallucinated CET1 ratio slips through a prompt change, appears in a client presentation, and contradicts the actual 10-K. In financial services this is not just an embarrassment — it can trigger a fiduciary liability claim. The eval harness we build in this notebook would have caught the hallucination by checking that the answer is supported by the retrieved context.

<br>

**What "measurement" means.** We need metrics that are (a) repeatable — the same inputs produce the same score, (b) sensitive — they detect the regressions you care about, and (c) cheap to compute — they can run on every commit. Unit evals satisfy all three; metric-based evals and LLM judges trade some cheapness for broader coverage.

## The Eval Stack

We organize evaluation into three tiers, each adding coverage at the cost of speed and complexity.

**Tier 1 — Unit evals.** Deterministic assertions on output structure: is the response valid JSON? Does it contain all required fields? Is the cited document index within the valid range? Unit evals run in milliseconds and need no LLM call — they are the cheapest safety net and should run on every change. We implement three examples: a JSON validity check, a required-fields check, and a citation range check.

<br>

**Tier 2 — Metric-based evals.** RAGAS computes four metrics that quantify different failure modes. **Faithfulness** measures whether every claim in the answer is entailed by the retrieved context — it catches hallucinations. **Answer relevance** measures whether the answer addresses the question — it catches off-topic responses. **Context recall** measures whether the retrieved set contains the passage needed to answer the question — it diagnoses retrieval failures. **Context precision** measures what fraction of the retrieved context is relevant — it diagnoses context noise.

<br>

**Tier 3 — LLM-as-judge.** When ground-truth answers are unavailable or too costly to write, we use a strong model (GPT-4o) to score outputs on a 1–5 rubric. This is the **G-Eval** pattern: chain-of-thought reasoning followed by a numeric score. It is slower and more expensive than unit evals but generalizes to open-ended quality dimensions — nuance, tone, completeness — that rule-based checks cannot capture.

:::{.callout-note}
Run Tier 1 on every commit, Tier 2 on every PR merge, and Tier 3 on every release. The cost structure makes this natural: unit evals are free, RAGAS requires a few embedding API calls, and LLM judges cost ~$0.01 per test case.

:::

We define three unit eval functions representing the most common structural checks:

In [ ]:
def check_valid_json(response: str) -> tuple[bool, str]:
    """Assert the response is parseable JSON."""
    try:
        json.loads(response)
        return True, "ok"
    except json.JSONDecodeError as e:
        return False, f"JSON parse error: {e}"


def check_required_fields(response_dict: dict, required: list[str]) -> tuple[bool, str]:
    """Assert all required keys are present in the response dict."""
    missing = [k for k in required if k not in response_dict]
    if missing:
        return False, f"Missing fields: {missing}"
    return True, "ok"


def check_citation_range(response: str, n_chunks: int) -> tuple[bool, str]:  # <1>
    """Assert all [N] citations in the response refer to valid chunk indices."""
    import re
    citations = [int(m) for m in re.findall(r'\[(\d+)\]', response)]
    invalid = [c for c in citations if c < 1 or c > n_chunks]
    if invalid:
        return False, f"Invalid citation indices: {invalid} (valid range: 1–{n_chunks})"
    return True, "ok"


# --- smoke tests ---
print(check_valid_json('{"risk_tier": "high", "score": 4}'))
print(check_valid_json('not json at all'))
print(check_required_fields({"risk_tier": "high"}, required=["risk_tier", "score"]))
print(check_citation_range("The CET1 ratio was 14.8% [1]. See also [7] and [12].", n_chunks=10))

1. We use a regex `\[(\d+)\]` to extract all citation markers from the response, then check each extracted index is within the valid chunk range. A citation to `[0]` or `[11]` when only 10 chunks were retrieved is a structural hallucination — the model invented a source reference that does not exist.

## RAGAS Metrics

RAGAS (Retrieval-Augmented Generation Assessment) defines four metrics that each probe a different failure mode. We state the key formulas before running the library.

**Faithfulness** decomposes the answer into atomic claims, then asks an LLM whether each claim is entailed by the context. If the answer makes $c$ claims and $v$ of them are supported:

$$\text{faithfulness} = \frac{v}{c}$$

A score of $1.0$ means every sentence in the answer is grounded in the retrieved context. A score below $0.8$ is a red flag for hallucination.

<br>

**Answer relevance** embeds both the answer and $n$ synthetic questions generated from the answer, then averages the cosine similarity between the answer embedding and each synthetic question embedding:

$$\text{answer\_relevance} = \frac{1}{n} \sum_{i=1}^{n} \text{sim}(\mathbf{e}_{q_i}, \mathbf{e}_{a})$$

An answer that addresses a different question than the one asked will score low.

<br>

**Context recall** compares each sentence in the ground-truth answer against the retrieved context and measures what fraction of those sentences are attributable to retrieved passages:

$$\text{context\_recall} = \frac{|\text{GT sentences attributable to context}|}{|\text{GT sentences}|}$$

A low context recall means the retriever failed: the answer it needed was not in the top-$k$ results.

We define the same 10-sentence financial corpus from [notebook 03](/courses/llm-eng/03-rag-concepts.html):

In [ ]:
CORPUS = [
    # Risk factors
    "Interest rate risk represents one of the most significant market risks facing "
    "the firm. A 100 basis point increase in interest rates would reduce the fair value "
    "of our fixed-rate debt portfolio by approximately $2.3 billion.",

    "Credit risk arises from the potential that a counterparty will fail to "
    "perform its obligations. We manage credit risk through diversification, "
    "collateral requirements, and credit limits by counterparty.",

    "Operational risk includes the risk of loss resulting from inadequate or "
    "failed internal processes, people, systems, or external events, including "
    "cybersecurity threats and technology failures.",

    # MD&A
    "Net revenues for the fiscal year were $47.4 billion, an increase of 8% "
    "compared to the prior year. The increase was driven primarily by higher "
    "net interest income reflecting the rising interest rate environment.",

    "Investment banking revenues decreased 23% to $6.1 billion, reflecting "
    "lower advisory fees amid reduced M&A activity and a challenging "
    "environment for equity and debt underwriting.",

    "Return on equity for the year was 12.4%, compared to 15.1% in the prior "
    "year. Book value per share increased to $312.50, up from $290.20.",

    # Capital and liquidity
    "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, "
    "well above the regulatory minimum of 4.5% and our internal target of 13%.",

    "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the "
    "regulatory requirement of 100%. Our high-quality liquid assets totaled "
    "$280 billion at year-end.",

    # Forward guidance
    "Looking ahead to fiscal 2025, management expects continued revenue growth "
    "in the range of 4-6%, supported by a stable rate environment and "
    "improving capital markets activity.",

    "We plan to return $8 billion to shareholders through dividends and share "
    "repurchases in fiscal 2025, subject to regulatory approval and market conditions.",
]

We build a 5-item eval dataset with question, answer, context, and ground-truth answer tuples drawn from the corpus:

In [ ]:
from datasets import Dataset

# Five QA pairs grounded in the CORPUS
eval_data = [
    {
        "question": "What is the CET1 capital ratio and how does it compare to the regulatory minimum?",
        "answer": "The CET1 capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and the internal target of 13%.",
        "contexts": [CORPUS[6]],
        "ground_truth": "The CET1 capital ratio was 14.8%, well above the regulatory minimum of 4.5% and the internal target of 13%.",
    },
    {
        "question": "How did investment banking revenues change year-over-year?",
        "answer": "Investment banking revenues decreased 23% to $6.1 billion, driven by lower advisory fees and reduced M&A activity.",
        "contexts": [CORPUS[4]],
        "ground_truth": "Investment banking revenues decreased 23% to $6.1 billion due to lower advisory fees and a challenging underwriting environment.",
    },
    {
        "question": "What is the company's liquidity coverage ratio?",
        "answer": "The LCR is 128%, exceeding the 100% regulatory requirement. High-quality liquid assets totalled $280 billion.",
        "contexts": [CORPUS[7]],
        "ground_truth": "The LCR is 128%, above the 100% regulatory minimum, with $280 billion in high-quality liquid assets.",
    },
    {
        "question": "What was the return on equity for the fiscal year?",
        "answer": "Return on equity was 12.4%, down from 15.1% in the prior year.",
        "contexts": [CORPUS[5]],
        "ground_truth": "ROE was 12.4%, compared to 15.1% in the prior year.",
    },
    {
        "question": "What revenue growth does management expect for fiscal 2025?",
        "answer": "Management expects revenue growth of 4-6% for fiscal 2025, supported by a stable rate environment.",
        "contexts": [CORPUS[8]],
        "ground_truth": "Management expects 4-6% revenue growth in fiscal 2025, supported by a stable rate environment and improving capital markets.",
    },
]

eval_dataset = Dataset.from_list(eval_data)
print(f"Eval dataset: {len(eval_dataset)} examples")
print(eval_dataset.column_names)

Running `ragas.evaluate()` on the clean dataset:

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

result_clean = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ChatOpenAI(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
)
print("=== Clean pipeline ===")
print(result_clean)

Now we deliberately degrade the pipeline by passing wrong context chunks — simulating a retrieval failure — and rerun RAGAS to confirm the scores drop:

In [ ]:
# Degrade: swap every context with an unrelated chunk
degraded_data = [
    {**row, "contexts": [CORPUS[(i + 3) % len(CORPUS)]]}
    for i, row in enumerate(eval_data)
]
degraded_dataset = Dataset.from_list(degraded_data)

result_degraded = evaluate(
    dataset=degraded_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ChatOpenAI(model="gpt-4o-mini"),
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
)
print("=== Degraded pipeline (wrong context) ===")
print(result_degraded)

print("\n--- Score comparison ---")
for metric in ["faithfulness", "answer_relevancy", "context_recall"]:
    clean_score = result_clean[metric]
    degraded_score = result_degraded[metric]
    print(f"{metric:20s}: clean={clean_score:.3f}  degraded={degraded_score:.3f}  delta={degraded_score - clean_score:+.3f}")

Context recall drops sharply when the wrong chunks are passed — confirming that RAGAS correctly detects retrieval failures. Faithfulness may also decrease since the model's answer no longer aligns with the supplied context.

## LLM-as-Judge

The **G-Eval** pattern uses a strong model as a scorer: we give it a rubric, a question, and an answer, and ask it to reason step by step before assigning a score from 1 to 5. Because the judge reasons before scoring, it is more reliable than asking for a score directly — the chain-of-thought anchors the numeric judgment to stated reasoning.

The rubric for financial Q&A is:
- **5**: Answer is factually correct, cites specific numbers from context, addresses the question completely.
- **4**: Answer is mostly correct with minor omissions.
- **3**: Answer is partially correct — gets the direction right but misses key figures.
- **2**: Answer is mostly wrong or misses the core of the question.
- **1**: Answer is factually incorrect, hallucinates numbers, or is irrelevant.

We implement `GEvalScorer` using Pydantic structured output to ensure we always get a numeric score back:

In [ ]:
class GEvalResult(BaseModel):
    reasoning: str
    score: int  # 1–5


class GEvalScorer:
    """Score (question, answer) pairs on a 1-5 rubric using an LLM judge."""

    RUBRIC = """
Score the answer to the financial question on a scale of 1-5:
5 = Factually correct, cites specific numbers from context, fully addresses the question.
4 = Mostly correct with minor omissions.
3 = Partially correct — right direction but missing key figures.
2 = Mostly wrong or misses the core question.
1 = Factually incorrect, hallucinates numbers, or irrelevant.

First reason step by step, then give a score.
"""

    def __init__(self, judge_model: str = "gpt-4o"):
        self._llm = LLMClient(model=judge_model, temperature=0.0)  # <1>

    def score_one(
        self,
        question: str,
        answer: str,
        context: str = "",
    ) -> GEvalResult:
        context_block = f"\n\nContext: {context}" if context else ""
        messages = [
            {"role": "system", "content": self.RUBRIC.strip()},
            {
                "role": "user",
                "content": f"Question: {question}{context_block}\n\nAnswer: {answer}",
            },
        ]
        return self._llm.complete(messages, response_format=GEvalResult)

    def score_batch(
        self,
        pairs: list[dict],  # each: {"question": ..., "answer": ..., "context": ...}
    ) -> list[GEvalResult]:
        return [self.score_one(**p) for p in pairs]  # <2>

1. We use `gpt-4o` as the judge — a stronger model than the pipeline under test (`gpt-4o-mini`). Using the same model as both generator and judge introduces bias: the model tends to rate its own outputs highly regardless of quality.
2. `score_batch` runs sequentially to keep the implementation simple. In production, parallelize with `asyncio.gather` to avoid waiting for each judge call serially.

We run the judge on three hand-crafted answers — a good answer, a partially correct one, and a hallucinated one — and verify the scores correlate with quality:

In [ ]:
scorer = GEvalScorer(judge_model="gpt-4o")

test_pairs = [
    {
        "question": "What is the CET1 capital ratio and how does it compare to the regulatory minimum?",
        "answer": "The CET1 capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and the internal target of 13%.",  # good
        "context": CORPUS[6],
    },
    {
        "question": "What is the CET1 capital ratio and how does it compare to the regulatory minimum?",
        "answer": "The CET1 ratio was around 15%, which is above the minimum requirement.",  # partially correct
        "context": CORPUS[6],
    },
    {
        "question": "What is the CET1 capital ratio and how does it compare to the regulatory minimum?",
        "answer": "The CET1 ratio was 11.2%, which is below the required 12% minimum — the firm is not compliant.",  # hallucinated
        "context": CORPUS[6],
    },
]

results = scorer.score_batch(test_pairs)
labels = ["good", "partial", "hallucinated"]
for label, result in zip(labels, results):
    print(f"[{label}] score={result.score}")
    print(f"  reasoning: {result.reasoning[:120]}...\n")

The judge assigns highest score to the answer that matches the context exactly, a middling score to the imprecise answer, and the lowest score to the hallucinated figures. This calibration is what makes LLM-as-judge useful: it can detect semantic-level quality differences that regex checks miss.

## Eval Harness

We integrate the three evaluation tiers into a single `EvalHarness` class. The harness stores test cases (each with a question, expected answer, context chunks, and the RAG pipeline under test), runs all three tiers, and returns a summary dictionary with pass rates and per-metric scores. This is the object you register in CI/CD and run on every prompt change.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable


@dataclass
class TestCase:
    question: str
    expected_answer: str
    context_chunks: list[str]
    section: str = "general"


class EvalHarness:
    """Three-tier evaluation harness for a RAG Q&A pipeline."""

    def __init__(self, pipeline_fn: Callable[[str, list[str]], str]):
        self._pipeline = pipeline_fn  # <1>
        self._cases: list[TestCase] = []
        self._scorer = GEvalScorer(judge_model="gpt-4o")

    def register(self, case: TestCase) -> None:
        self._cases.append(case)

    def _run_pipeline(self, case: TestCase) -> str:
        """Generate an answer for a test case using the registered pipeline."""
        return self._pipeline(case.question, case.context_chunks)

    def run(self) -> dict:
        """Run all tiers on all registered test cases. Returns a summary dict."""
        answers = [self._run_pipeline(c) for c in self._cases]

        # Tier 1: unit evals
        unit_passes = []
        for ans, case in zip(answers, self._cases):
            n = len(case.context_chunks)
            ok, _ = check_citation_range(ans, n)
            unit_passes.append(ok)

        unit_pass_rate = sum(unit_passes) / max(len(unit_passes), 1)

        # Tier 3: LLM-as-judge scores
        judge_pairs = [
            {
                "question": c.question,
                "answer": ans,
                "context": " ".join(c.context_chunks),
            }
            for c, ans in zip(self._cases, answers)
        ]
        judge_results = self._scorer.score_batch(judge_pairs)  # <2>
        scores = [r.score for r in judge_results]
        avg_judge_score = sum(scores) / max(len(scores), 1)

        return {
            "n_cases": len(self._cases),
            "unit_pass_rate": unit_pass_rate,
            "judge_scores": scores,
            "avg_judge_score": avg_judge_score,
            "answers": answers,
        }

1. The harness is pipeline-agnostic: `pipeline_fn` takes a question and a list of context strings and returns an answer string. This makes it trivial to swap between prompt versions, models, or RAG configurations without touching the eval logic.
2. Judge calls are the most expensive part of the harness — each costs roughly $0.005 at GPT-4o pricing. For large eval sets, run the judge on a random 20% sample and apply unit evals to 100%.

We define two pipeline variants — a strong and a weak system prompt — and demonstrate that the harness detects the regression:

In [ ]:
def make_pipeline(system_prompt: str) -> Callable[[str, list[str]], str]:
    """Return a pipeline function backed by the given system prompt."""
    def pipeline(question: str, context_chunks: list[str]) -> str:
        context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ]
        return llm.complete(messages)
    return pipeline


STRONG_PROMPT = (
    "You are a financial analyst assistant. "
    "Answer using ONLY the provided context. "
    "Cite sources using [N] notation. "
    "Include specific numbers. "
    "If the context is insufficient, say so explicitly."
)

WEAK_PROMPT = "Answer the question helpfully."  # deliberately vague

# Register three test cases from the corpus
cases = [
    TestCase(
        question="What is the CET1 capital ratio?",
        expected_answer="14.8%, above the 4.5% regulatory minimum.",
        context_chunks=[CORPUS[6]],
    ),
    TestCase(
        question="What were net revenues for the fiscal year?",
        expected_answer="$47.4 billion, up 8% year-over-year.",
        context_chunks=[CORPUS[3]],
    ),
    TestCase(
        question="What is the outlook for fiscal 2025 revenue growth?",
        expected_answer="4-6% growth expected, supported by a stable rate environment.",
        context_chunks=[CORPUS[8]],
    ),
]

print("=== Strong prompt ===")
harness_strong = EvalHarness(pipeline_fn=make_pipeline(STRONG_PROMPT))
for c in cases:
    harness_strong.register(c)
result_strong = harness_strong.run()
print(f"unit_pass_rate:   {result_strong['unit_pass_rate']:.2f}")
print(f"avg_judge_score:  {result_strong['avg_judge_score']:.2f}")
print(f"judge_scores:     {result_strong['judge_scores']}")

print("\n=== Weak prompt (regression) ===")
harness_weak = EvalHarness(pipeline_fn=make_pipeline(WEAK_PROMPT))
for c in cases:
    harness_weak.register(c)
result_weak = harness_weak.run()
print(f"unit_pass_rate:   {result_weak['unit_pass_rate']:.2f}")
print(f"avg_judge_score:  {result_weak['avg_judge_score']:.2f}")
print(f"judge_scores:     {result_weak['judge_scores']}")

print(f"\nRegression detected: {result_strong['avg_judge_score'] - result_weak['avg_judge_score']:+.2f} points")

The harness reports a lower average judge score for the weak prompt — the regression is visible and quantified. In CI/CD, this comparison runs automatically: if `avg_judge_score` drops more than 0.5 points relative to the baseline, the deploy is blocked.

:::{.callout-caution}
LLM judges are stochastic. Run with `temperature=0.0` and consider averaging over 3 independent judge calls per test case for critical evals — the variance on a single call can be ±0.5 points on a 1–5 scale.

:::

## Exercises

1. **Add a source-citation unit eval.** Write a function `check_has_citation(response: str) -> tuple[bool, str]` that returns `(False, "missing citation")` if the answer contains no `[N]` markers. Register it in `EvalHarness.run` alongside `check_citation_range`.

2. **Add a corpus test case and verify the harness catches bad retrieval.** Define a `TestCase` with the correct context chunk for the LCR question (`CORPUS[7]`), then run the harness with the wrong context chunk (`CORPUS[0]`) substituted. Confirm that `avg_judge_score` drops relative to the correct-context run.

3. **Modify the judge rubric to penalise invented numbers.** Add a criterion to `GEvalScorer.RUBRIC`: "Deduct 2 points if the answer mentions any specific figure (percentage, dollar amount) that does not appear verbatim in the provided context." Re-run the hallucinated test case from the G-Eval section and verify the score decreases further.

---

$\blacksquare$